# Full Pipeline Demo: Wardrobe Outfit Recommendation System

This notebook demonstrates the complete end-to-end pipeline for outfit recommendations.

**Features:**
- Upload your own wardrobe images
- Automatic garment classification
- **Multi-color pattern detection** (solid, two-tone, multi-color)
- Color and attribute extraction
- Outfit generation with explanations
- Interactive outfit building

In [ ]:
# Setup
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from IPython.display import display, HTML
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Import all modules
from src.recognition.classifier import GarmentClassifier, get_device
from src.attributes.color_extractor import ColorExtractor, ColorHarmonyAnalyzer
from src.attributes.pipeline import AttributePipeline, GarmentAttributes
from src.attributes.multicolor import (
    ColorPattern, 
    ColorClassification,
    EnhancedColorExtractor,
    EnhancedHarmonyAnalyzer
)
from src.compatibility.model import SiameseCompatibilityNet
from src.compatibility.scorer import CompatibilityScorer
from src.generation.generator import OutfitGenerator, Wardrobe, WardrobeItem, OutfitTemplate
from src.generation.explainer import OutfitExplainer, QuickExplainer

print("All modules imported successfully!")
print(f"Device: {get_device()}")

## 1. Initialize System

In [ ]:
# Paths
MODEL_DIR = Path('../models')
SAMPLE_DIR = Path('../data/samples')
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

device = get_device()

# Load or create classifier
print("Loading models...")
classifier_path = MODEL_DIR / 'garment_classifier.pth'
if classifier_path.exists():
    classifier = GarmentClassifier.load(classifier_path)
    print("  [x] Loaded trained garment classifier")
else:
    classifier = GarmentClassifier(backbone="efficientnet_b0", pretrained=True)
    classifier.to(device)
    classifier.device = device
    print("  [!] Using pretrained backbone (run notebook 01 to train)")

# Enhanced color extractor with multi-color classification
color_extractor = EnhancedColorExtractor(
    n_colors=5, 
    color_space='LAB',
    solid_threshold=0.85,
    two_tone_threshold=0.50
)
print("  [x] Enhanced color extractor ready (with multi-color classification)")

# Compatibility model
compat_path = MODEL_DIR / 'compatibility_model.pth'
if compat_path.exists():
    compat_model = SiameseCompatibilityNet.load(str(compat_path))
    print("  [x] Loaded trained compatibility model")
else:
    compat_model = SiameseCompatibilityNet()
    compat_model.to(device)
    print("  [!] Using untrained compatibility model (run notebook 03 to train)")

# Create pipeline components with multi-color enabled
pipeline = AttributePipeline(
    classifier=classifier, 
    color_extractor=color_extractor,
    enable_multicolor=True
)
scorer = CompatibilityScorer(
    model=compat_model, 
    classifier=classifier, 
    color_extractor=color_extractor
)
generator = OutfitGenerator(scorer=scorer, min_compatibility=0.3)
enhanced_harmony = EnhancedHarmonyAnalyzer()
explainer = OutfitExplainer(color_analyzer=ColorHarmonyAnalyzer())

print("\nSystem ready with multi-color classification!")

## 2. Demo: Multi-Color Classification

The system now classifies garments into three color pattern types:
- **Solid**: Primary color >= 85%
- **Two-tone**: Primary color 50-85% with significant secondary
- **Multi-color**: Primary < 50% or 3+ significant colors

In [ ]:
# Create demo images with different color patterns
def create_solid_image(color_rgb, size=224):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[:, :] = color_rgb
    return img

def create_two_tone_image(color1, color2, size=224):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[:size//2, :] = color1
    img[size//2:, :] = color2
    return img

def create_striped_image(colors, stripe_width=30, size=224):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    for i in range(0, size, stripe_width):
        color_idx = (i // stripe_width) % len(colors)
        img[i:i+stripe_width, :] = colors[color_idx]
    return img

# Create test images
demo_images = {
    "Solid Navy Shirt": create_solid_image([30, 50, 100]),
    "Two-Tone (Navy/White)": create_two_tone_image([30, 50, 100], [255, 255, 255]),
    "Multi-Color Stripes": create_striped_image([[200, 50, 50], [255, 255, 255], [30, 50, 100]])
}

print("Demo images created!")

In [ ]:
# Process each demo image through the pipeline
demo_results = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, img) in enumerate(demo_images.items()):
    # Process through pipeline
    attrs = pipeline.process(img, item_id=name, override_category='tops')
    demo_results[name] = attrs
    
    # Display
    ax = axes[idx]
    ax.imshow(img)
    
    # Build title with color info
    pattern = attrs.color_pattern.value if attrs.color_pattern else "unknown"
    color_summary = attrs.get_color_summary()
    
    ax.set_title(f"{name}\n\nPattern: {pattern.upper()}\n{color_summary}", fontsize=10)
    ax.axis('off')

plt.suptitle("Multi-Color Classification Demo", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print detailed results
print("\nDetailed Classification Results:")
print("=" * 60)
for name, attrs in demo_results.items():
    print(f"\n{name}:")
    print(f"  Pattern: {attrs.color_pattern.value if attrs.color_pattern else 'N/A'}")
    print(f"  is_solid(): {attrs.is_solid()}")
    print(f"  is_two_tone(): {attrs.is_two_tone()}")
    print(f"  is_multicolor(): {attrs.is_multicolor()}")
    print(f"  Color Summary: {attrs.get_color_summary()}")

## 3. Enhanced Harmony Analysis

The enhanced harmony analyzer considers the full color pattern when scoring compatibility.

In [ ]:
# Test enhanced harmony analysis between different pattern types
print("Enhanced Harmony Analysis:")
print("=" * 60)

test_pairs = [
    ("Solid Navy Shirt", "Solid Navy Shirt"),  # Same solid
    ("Solid Navy Shirt", "Two-Tone (Navy/White)"),  # Solid + two-tone with matching color
    ("Two-Tone (Navy/White)", "Multi-Color Stripes"),  # Two patterns
]

for name1, name2 in test_pairs:
    attrs1 = demo_results[name1]
    attrs2 = demo_results[name2]
    
    if attrs1.color_classification and attrs2.color_classification:
        result = enhanced_harmony.analyze_harmony_enhanced(
            attrs1.color_classification,
            attrs2.color_classification
        )
        
        print(f"\n{name1} + {name2}:")
        print(f"  Pattern combo: {result['pattern_combination']}")
        print(f"  Harmony score: {result['score']:.2f}")
        print(f"  Note: {result['pattern_note']}")

## 4. Create Sample Wardrobe

Let's create a sample wardrobe with various color patterns.

In [ ]:
# Create a diverse wardrobe with different color patterns
wardrobe_items = {
    # Tops - mix of patterns
    "White T-Shirt": (create_solid_image([255, 255, 255]), "tops"),
    "Navy Polo": (create_solid_image([30, 50, 100]), "tops"),
    "Striped Shirt": (create_striped_image([[255, 255, 255], [50, 100, 150]], 20), "tops"),
    "Black Top": (create_solid_image([30, 30, 30]), "tops"),
    
    # Bottoms
    "Dark Jeans": (create_solid_image([40, 50, 80]), "bottoms"),
    "Khaki Pants": (create_solid_image([195, 176, 145]), "bottoms"),
    "Black Pants": (create_solid_image([25, 25, 25]), "bottoms"),
    
    # Shoes
    "White Sneakers": (create_solid_image([250, 250, 250]), "shoes"),
    "Brown Boots": (create_solid_image([139, 90, 43]), "shoes"),
}

# Process all items
wardrobe = Wardrobe()
processed_items = {}

print("Processing wardrobe items...\n")
for name, (img, category) in wardrobe_items.items():
    attrs = pipeline.process(img, item_id=name, override_category=category)
    
    # Create WardrobeItem
    item = WardrobeItem(
        item_id=name,
        category=category,
        visual_features=attrs.visual_embedding,
        color_features=attrs.color_vector,
        dominant_colors=attrs.dominant_colors,
        metadata={
            'name': name,
            'color_pattern': attrs.color_pattern.value if attrs.color_pattern else None,
            'color_summary': attrs.get_color_summary()
        }
    )
    
    wardrobe.add_item(item)
    processed_items[name] = (attrs, item)
    
    pattern = attrs.color_pattern.value if attrs.color_pattern else "unknown"
    print(f"  {name}: {category} | {pattern} | {attrs.get_color_summary()}")

print(f"\nWardrobe ready with {len(wardrobe)} items!")

In [ ]:
# Visualize wardrobe by color pattern
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for idx, (name, (img, category)) in enumerate(wardrobe_items.items()):
    if idx >= len(axes):
        break
    
    ax = axes[idx]
    ax.imshow(img)
    
    attrs, _ = processed_items[name]
    pattern = attrs.color_pattern.value if attrs.color_pattern else "?"
    
    # Color-code border by pattern type
    border_colors = {
        'solid': 'green',
        'two-tone': 'orange', 
        'multi-color': 'red'
    }
    border = border_colors.get(pattern, 'gray')
    for spine in ax.spines.values():
        spine.set_edgecolor(border)
        spine.set_linewidth(3)
    
    ax.set_title(f"{name}\n[{pattern}]", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

# Hide unused axes
for idx in range(len(wardrobe_items), len(axes)):
    axes[idx].axis('off')

plt.suptitle("Wardrobe Items by Color Pattern\n(Green=Solid, Orange=Two-Tone, Red=Multi-Color)", 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Generate Outfits with Pattern-Aware Scoring

In [ ]:
# Generate outfit recommendations
print("Generating outfit recommendations...\n")

template = OutfitTemplate("casual", ["tops", "bottoms", "shoes"])

outfits = generator.generate_outfits(
    wardrobe=wardrobe,
    template=template,
    max_outfits=5
)

print(f"Generated {len(outfits)} outfit recommendations!")

In [ ]:
# Display outfits with color pattern info
def display_outfit_with_patterns(outfit, rank, wardrobe_items):
    """Display an outfit with color pattern information."""
    n_items = len(outfit.items)
    fig, axes = plt.subplots(1, n_items + 1, figsize=(4*(n_items+1), 4),
                            gridspec_kw={'width_ratios': [1]*n_items + [1.5]})
    
    pattern_info = []
    
    for i, item in enumerate(outfit.items):
        # Get the image
        img_data = wardrobe_items.get(item.item_id)
        if img_data:
            img = img_data[0]
            axes[i].imshow(img)
        elif item.dominant_colors:
            color = np.array(item.dominant_colors[0].rgb) / 255
            block = np.ones((100, 100, 3)) * color
            axes[i].imshow(block)
        
        # Get pattern info from metadata
        pattern = item.metadata.get('color_pattern', 'unknown')
        color_summary = item.metadata.get('color_summary', '')
        pattern_info.append((item.item_id, pattern))
        
        axes[i].set_title(f"{item.item_id}\n({item.category})\n[{pattern}]", fontsize=9)
        axes[i].axis('off')
    
    # Analysis panel
    text = f"Score: {outfit.score:.0%}\n\n"
    text += "Color Patterns:\n"
    for name, pattern in pattern_info:
        text += f"  - {name}: {pattern}\n"
    
    # Pattern combination advice
    patterns = [p for _, p in pattern_info]
    if all(p == 'solid' for p in patterns):
        text += "\nAdvice: All solid colors - classic and safe!"
    elif patterns.count('solid') >= 2:
        text += "\nAdvice: Good balance of solid with accent pattern."
    elif patterns.count('multi-color') >= 2:
        text += "\nAdvice: Multiple patterns - ensure colors coordinate."
    
    axes[-1].text(0.05, 0.95, text, transform=axes[-1].transAxes,
                  fontsize=10, verticalalignment='top', family='monospace',
                  bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    axes[-1].set_title("Analysis", fontsize=10)
    axes[-1].axis('off')
    
    fig.suptitle(f"Outfit #{rank}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Display top outfits
for i, outfit in enumerate(outfits[:5], 1):
    display_outfit_with_patterns(outfit, i, wardrobe_items)
    print()

## 6. Pattern-Based Filtering

Filter outfits by color pattern rules.

In [ ]:
# Find outfits with specific pattern combinations
def filter_outfits_by_pattern(outfits, rule='all_solid'):
    """Filter outfits by pattern rules."""
    filtered = []
    
    for outfit in outfits:
        patterns = [item.metadata.get('color_pattern', 'unknown') for item in outfit.items]
        
        if rule == 'all_solid':
            if all(p == 'solid' for p in patterns):
                filtered.append(outfit)
        elif rule == 'has_pattern':
            if any(p in ['two-tone', 'multi-color'] for p in patterns):
                filtered.append(outfit)
        elif rule == 'one_pattern_max':
            pattern_count = sum(1 for p in patterns if p in ['two-tone', 'multi-color'])
            if pattern_count <= 1:
                filtered.append(outfit)
    
    return filtered

# Example: Find all-solid outfits
all_outfits = generator.generate_outfits(wardrobe, template, max_outfits=20)

solid_outfits = filter_outfits_by_pattern(all_outfits, 'all_solid')
pattern_outfits = filter_outfits_by_pattern(all_outfits, 'has_pattern')

print(f"Total outfits generated: {len(all_outfits)}")
print(f"All-solid outfits: {len(solid_outfits)}")
print(f"Outfits with patterns: {len(pattern_outfits)}")

## 7. Wardrobe Summary with Color Analysis

In [ ]:
# Generate wardrobe summary with color pattern breakdown
print("=" * 60)
print("WARDROBE SUMMARY")
print("=" * 60)

# Count by category
counts = wardrobe.get_category_counts()
print(f"\nTotal Items: {len(wardrobe)}")
print("\nBy Category:")
for cat, count in counts.items():
    print(f"  {cat}: {count}")

# Count by color pattern
pattern_counts = {'solid': 0, 'two-tone': 0, 'multi-color': 0, 'unknown': 0}
for name, (attrs, item) in processed_items.items():
    pattern = attrs.color_pattern.value if attrs.color_pattern else 'unknown'
    pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

print("\nBy Color Pattern:")
for pattern, count in pattern_counts.items():
    if count > 0:
        pct = count / len(wardrobe) * 100
        print(f"  {pattern}: {count} ({pct:.0f}%)")

# Most common colors
all_colors = []
for name, (attrs, item) in processed_items.items():
    if attrs.dominant_colors:
        all_colors.append(attrs.dominant_colors[0].name)

color_counts = Counter(all_colors).most_common(5)

print("\nMost Common Colors:")
for color, count in color_counts:
    print(f"  {color}: {count}")

# Outfit potential
tops = counts.get('tops', 0)
bottoms = counts.get('bottoms', 0)
shoes = counts.get('shoes', 0)

print(f"\nOutfit Combinations: {tops * bottoms * max(shoes, 1):,}")

## Summary

This demo showcased:

1. **Multi-Color Classification** - Automatically categorizes garments as:
   - Solid (>= 85% single color)
   - Two-tone (50-85% primary with significant secondary)
   - Multi-color (< 50% primary or 3+ colors)

2. **Enhanced Harmony Analysis** - Better outfit scoring that considers:
   - Pattern combinations (solid + solid, solid + pattern, etc.)
   - Shared colors between multi-color items
   - Pattern clash detection

3. **Pattern-Aware Recommendations** - Filter and sort outfits by:
   - All-solid combinations
   - Maximum one pattern rule
   - Color coordination scoring

**Next Steps:**
- Add your own images to `data/samples/` to test on real clothing
- Train the classifier (notebook 01) for better category accuracy
- Train the compatibility model (notebook 03) for better outfit scoring
- Explore the multi-color classification in detail (notebook 06)